<a href="https://colab.research.google.com/github/lcbjrrr/genai/blob/main/AgentsGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
!pip install --upgrade langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.6
    Uninstalling langgraph-1.1.6:
      Successfully uninstalled langgraph-1.1.6


# AI Agents

## ChatGPT Agent

The agent receives information from the environment through a perception process, which feeds the current state or user input into the LLM. Inside the agent boundary, the LLM engages in a planning stage, which is essentially the model’s internal reasoning process; here, it evaluates the situation, considers available information, and decides which tool or action is appropriate to use next.

This reasoning-driven planning stage is crucial because it allows the LLM to choose the correct tool from its toolbox to achieve its goal. After planning, the LLM produces an action that affects the environment, such as invoking a tool, sending a response, or modifying system state. The environment then changes, and this updated state flows back into the perception pathway, creating a continuous cycle of perceiving, reasoning, acting, and learning that defines the behavior of an autonomous AI agent.

This is also known as an **ReAct** agent (short for ***“Reason*** + ***Act”***) is an AI agent that alternates between reasoning steps and actions, using its internal thought process to decide when and how to call tools, observe the results, and iteratively refine its response.

![](https://pbs.twimg.com/media/HGR7_xhbQAAoNfu?format=jpg&name=small)

## Tools and an Agent

A tool is an external capability, such as a function, API call, database query, calculator, or code executor that the LLM can invoke to accomplish tasks it cannot solve through language generation alone, and in LangChain these tools are exposed to the model as callable functions described with schemas so the LLM can reason about them and select the correct one during its planning stage.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
import os

def state_capital_f(state) :
  state_capital_dict = {
      "MI":"Lansing",
      "FL":"Tallahassee",
      "CA":"Sacramento",
      "TX":"Austin",
      "NY":"Albany"}
  print("------>",state,state_capital_dict[state])
  return state_capital_dict[state]

capitals_tool = Tool(name="state_capitals",func=state_capital_f,
    description="This tool can identify the most up-to-date capital of each state")

state_capital_f("MI")

------> MI Lansing


'Lansing'

The LLM identifies which tool to use because LangChain injects each tool’s name, description, and input schema directly into the model’s prompt, allowing the model to reason over them. When the user asks a question, the LLM compares the query with the available tool descriptions and decides, through its own reasoning step, whether a tool is appropriate. If the query matches what a tool is designed to do, the model generates a structured tool call, LangChain executes it, and the result is returned to the model to produce the final answer.

In [ ]:
# pip install -U langchain langchain-openai
KEY='xxx'
os.environ["OPENAI_API_KEY"] = KEY
chat_gpt = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

agent = create_agent(model=chat_gpt,
    tools=[capitals_tool],
    system_prompt="You are a US based newscaster.")

def invoke_agent(prompt):
  response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
  return response['messages'][-1].content

invoke_agent("What is the CA capital?")

------> MI Lansing


'The capital of Michigan (MI) is Lansing.'

Here is a simply a function that returns a city’s historical average temperature in Fahrenheit, and when registered as a LangChain tool its name and description allow the LLM to recognize it as the tool to call whenever the user asks for temperature information about a supported city.

In [ ]:
def tempature_f(city) :
  temp_dict = {
      "Lansing":50,
      "Tallahassee":90,
      "Sacramento":85,
      "Austin":80,
      "Albany":45}
  print("=====>",city,temp_dict[city])
  return temp_dict[city]


temp_tool = Tool(name="cities_average_temperature",func=tempature_f,
        description="This tool can tell the historical average temperature for different cities")

tempature_f("Lansing")

=====> Lansing 50


50

Let's adjust our agento to use the LLM model and is equipped with both the capitals lookup tool and the temperature lookup tool, while operating under the system instruction to behave as a US-based newscaster.

In [ ]:
KEY='xxx'
os.environ["OPENAI_API_KEY"] = KEY
chat_gpt = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

agent = create_agent(model=chat_gpt,
    tools=[capitals_tool,temp_tool],
    system_prompt="You a US based newscaster")
invoke_agent("What is the Michigan capital? And what is the average temperature in there?")

=====> Lansing 50
------> MI Lansing


'The capital of Michigan (MI) is Lansing. The historical average temperature in Lansing is around 50 degrees Fahrenheit.'

Including a memory component, it gives the agent the ability to retain information across turns, meaning it can remember previous user questions, facts it has already retrieved, and details from earlier interactions. This enables more coherent multi-step conversations, lets the agent avoid repeating tool calls unnecessarily, and allows it to behave more like a consistent, context-aware assistant rather than treating every query as isolated.

The thread specifies which conversation thread the request belongs to, allowing the agent to retrieve and update the correct memory so multiple interactions are linked within the same ongoing context.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
chat_gpt = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

agent = create_agent(
    model=chat_gpt,
    tools=[capitals_tool,temp_tool],
    checkpointer=InMemorySaver(),
    system_prompt="You a US based newscaster")

def invoke_agent(prompt,thread=""):
  response = agent.invoke({"messages": [{"role": "user", "content": prompt}]},
                          {"configurable": {"thread_id": thread}})
  return response['messages'][-1].content

msg = input('> Ask something (type .exit to leave) ')
while msg!=".exit":
  print(invoke_agent(msg))
  msg = input('> Ask something (type .exit to leave) ')
print("Bye!!!")
#"What is the Michigan capital? And what is the average temperature in there?"
#"What is the California capital? And what is the average temperature in there?"
#"What is the average temperature difference between these two cities?"

> Ask something (type .exit to leave) hi
Hello! How can I assist you today?
> Ask something (type .exit to leave).exit
Bye!!!


## AI Agent Example

This code create a more compreenshive example with the **Perception of the Environment**.

PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;

PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;

PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;
    
\n\nHere is the table name ***teams***: *Stores the teams names and their nicknames*.\n    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;\n\nHere is the table name ***players***: *Contains the main players along their numbers for every team*.\n    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;\n\nHere is the table name ***coaches***: *Holds the record of the head coach for each team*.\n    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;

In [ ]:
import sqlite3
from langchain_core.tools import Tool
TABLES = {'teams':'Stores the teams' names and their nicknames',
          'players':'Contains the main players along with their numbers for every team',
          'coaches':'Holds the record of the head coach for each team'}
def get_table_context(db,table_name, table_description):
  conn = sqlite3.connect(db)
  cursor = conn.cursor()
  print(f"PRAGMA table_info({table_name});")
  cursor.execute(f"PRAGMA table_info({table_name});")
  schema = cursor.fetchall()
  columns_str = ""
  for col in schema:
    columns_str += f" **{col[1]}**: {col[2]};"
  context = f"""Here is the table name ***{table_name}***: *{table_description}*.
    Here are the columns of the {table_name}: {columns_str}"""
  print('---->',context)
  return context

def inspect_db_f(x=""):
  db='teams.db'
  table_context = ""
  for name, desc in TABLES.items():
    table_context += '\n\n'+ get_table_context(db,name,desc)
  return table_context


inspect_db_tool = Tool(name="inspect_db_tool",func=inspect_db_f,
        description="This tool can inspect a database and retrieve the table names and their columns")

inspect_db_f()

PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;
PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;
PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;


'\n\nHere is the table name ***teams***: *Stores the teams names and their nicknames*.\n    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;\n\nHere is the table name ***players***: *Contains the main players along their numbers for every team*.\n    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;\n\nHere is the table name ***coaches***: *Holds the record of the head coach for each team*.\n    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;'

Here is another tool that connects to a local SQLite database, executes whatever SQL query the LLM provides, and returns the results as a pandas DataFrame.

In [ ]:
import re
import sqlite3
import pandas as pd
from langchain_core.tools import Tool

def execute_query_f(query):
  DBNAME='teams.db'
  print('======>',query)
  conn = sqlite3.connect(DBNAME)
  cursor = conn.cursor()
  cursor.execute(query)
  return pd.DataFrame(cursor.fetchall())

execute_query_tool = Tool(name="execute_query_tool",func=execute_query_f,
        description="This tool can execute a SQL query in a database.")

execute_query_f('SELECT * FROM teams')

======> SELECT * FROM teams


,0,1,2
0,1,Troy,Trojans
1,2,Old Dominion,Monarchs
2,3,Louisiana,Ragin Cajuns
3,4,Missouri St.,Bears
4,5,Kennesaw St.,Owls
...,...,...,...
76,77,Utah,Utes
77,78,Rice,Owls
78,79,Arizona,Wildcats
79,80,Wake Forest,Demon Deacons


Let's create the agent with the two tools and with memmory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os
KEY='sk-xxx'
os.environ["OPENAI_API_KEY"] = KEY
chat_gpt = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

agent = create_agent(
    model=chat_gpt,
    tools=[inspect_db_tool,execute_query_tool],
    checkpointer=InMemorySaver(),
    system_prompt="With tool support, you can inspect databases to map their table structure. Based on your general SQL knowledge, and once you have inspected the database, create a final query. Run these queries using the tool that can execute queries over this database.")

def invoke_agent(prompt,thread=""):
  response = agent.invoke({"messages": [{"role": "user", "content": prompt}]},
                          {"configurable": {"thread_id": thread}})
  return response['messages'][-1].content

invoke_agent('What are the team names?')

PRAGMA table_info(teams);
----> Here is the table name ***teams***: *Stores the teams names and their nicknames*.
    Here are the columns of the teams:  **team_id**: INTEGER; **team**: TEXT; **nickname**: TEXT;
PRAGMA table_info(players);
----> Here is the table name ***players***: *Contains the main players along their numbers for every team*.
    Here are the columns of the players:  **player_id**: INTEGER; **player_name**: TEXT; **number**: INTEGER; **team_id**: INTEGER;
PRAGMA table_info(coaches);
----> Here is the table name ***coaches***: *Holds the record of the head coach for each team*.
    Here are the columns of the coaches:  **coach_id**: INTEGER; **coach_name**: TEXT; **team_id**: INTEGER;
======> SELECT team FROM teams;


'The team names are available in the "teams" table. Here are some of the team names: Troy, Old Dominion, Louisiana, Missouri St., Kennesaw St., Utah, Rice, Arizona, Wake Forest, Navy, and many more. If you want, I can provide the complete list of team names. Would you like me to do that?'

The agent perceives the environment by receiving external events, in this case, live or recently completed sports games—through an API call. When your code fetches the game results and passes them into invoke_agent(). The agent treats that data as its perception of the environment, allowing it to react to real-world changes like final scores and generate an appropriate response.

In [ ]:
import requests
import json
api_url = "https://ncaa-api.henrygd.me/scoreboard/football/fbs/all-conf/2025"
response = requests.get(api_url)
data = response.json()
game = data['games'][0]['game']
home_team = game['home']['names']['short']
away_team = game['away']['names']['short']
home_score = game['home']['score']
away_score = game['away']['score']

invoke_agent(f"Find in the database the nicknames of these teams {away_team} and {home_team}. Considering that the score was {away_score} x {home_score}, create a message to be posted on social media mentioning the teams' mascots. Return only the post, no intros or questions.")

======> SELECT nickname FROM teams WHERE team IN ('Army West Point', 'Navy');


'In an intense showdown, the Black Knights edged out the Mids with a nail-biting score of 16 to 17! What a game! #BlackKnights #Mids #EpicBattle'

The LLM breaks the prompt into steps during its reasoning phase and then selects the appropriate tools to execute each step. First, it infers which team won from the score and decides it needs the team’s ID, so it calls the SQL tool with a query like `SELECT team_id FROM teams WHERE team = 'Navy'`. After getting the team ID, it reasons that it now needs a player from that team, triggering a second call to the same database tool with `SELECT player_name FROM players WHERE team_id = 81 LIMIT 1`. Once it receives the player name, the LLM stops calling tools and generates the final congratulatory message using the gathered information.


In [ ]:
invoke_agent(f"Considering the score {away_team} {away_score} x {home_score} {home_team}, find in the database the one player from the winning team and write a congratulatory message to him. Return only the message, no intros or questions.")")

======> SELECT player_name FROM players WHERE team_id = (SELECT team_id FROM teams WHERE team = 'Navy') LIMIT 1;


'Congratulations to Noah Fifita from the Navy for the outstanding performance that helped secure the victory with a score of 17 to 16! Your hard work and dedication truly made the difference! #Navy #Victory #MVP'

The prompt produces those results because the LLM reasons that it must retrieve information about each team’s coach, so it decides to call your **SQL execution tool** multiple times. First, it uses the tool to look up each team’s `team_id` in the **teams** table (`SELECT team_id FROM teams WHERE team = ...`). Then, using those IDs, it calls the same tool again to fetch the coach names from the **coaches** table (`SELECT coach_name FROM coaches WHERE team_id = ...`). After the tool returns the data, the LLL uses its internal knowledge to generate short biographical sentences based on the retrieved coach names.


In [ ]:
invoke_agent(f"Considering that {away_team} played against {home_team}, find in the database the names of the coaches and see what you already know about them, and create one sentence each with a mini biography of each one. Return only the two sentences, no intros or questions.")


======> SELECT coach_name FROM coaches WHERE team_id = (SELECT team_id FROM teams WHERE team = 'Army West Point');
======> SELECT coach_name FROM coaches WHERE team_id = (SELECT team_id FROM teams WHERE team = 'Navy');


'Curt Cignetti is a seasoned football coach known for his strategic acumen and leadership, currently leading the Army West Point team. Brent Brennan is a dedicated coach with a strong focus on player development and team cohesion, serving as the head coach for the Navy team.'

Simulating the example of a game result that did not really came from a real event...

In [ ]:

home_team = 'Navy'
away_team = 'Troy'
home_score = 20
away_score = 11

invoke_agent(f"Find the mascots of these teams in the database: {away_team} and {home_team}. Considering that the score was {away_score} x {home_score}, create a message to be posted on social media mentioning the teams' mascots. Return only the post, no intros or questions.")

======> SELECT nickname FROM teams WHERE team IN ('Troy', 'Navy');


'The Trojans fought hard but the Mids dominated the field with a final score of 11 to 20! What a powerful performance! #Trojans #Mids #GameDayVictory'

... to test the memmory capability, in addition to all the tools call.

In [ ]:
team = 'Navy'
invoke_agent(f'Create a short paragraph about {team}, including its players, coach, and nickname. Also, list its most recent scores that you know of. Return only the paragraph, no intros or questions.')

======> SELECT player_name FROM players WHERE team_id = (SELECT team_id FROM teams WHERE team = 'Navy') LIMIT 3;


'The Navy team, known as the Mids, is led by coach Brent Brennan and features talented players like Noah Fifita. They have recently showcased strong performances with scores including a 17-16 victory against Army West Point and a 20-11 win over Troy.'

## APPENDIX



```
import sqlite3
conn = sqlite3.connect('team.db')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE teams (
        team_id INTEGER PRIMARY KEY AUTOINCREMENT,
        team TEXT,
        mascot TEXT)''')

cursor.execute('''CREATE TABLE players (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_name TEXT NOT NULL,
        number INTEGER,
        team_id INTEGER,
        FOREIGN KEY (team_id) REFERENCES teams(team_id))''')

cursor.execute('''CREATE TABLE coaches (
        coach_id INTEGER PRIMARY KEY AUTOINCREMENT,
        coach_name TEXT NOT NULL,
        team_id INTEGER,
        FOREIGN KEY (team_id) REFERENCES teams(team_id))''')

insert_statements = [
    "INSERT INTO teams (team,nickname) values ('Troy','Trojans')",
    "INSERT INTO teams (team,nickname) values ('Old Dominion','Monarchs')",
    "INSERT INTO teams (team,nickname) values ('Louisiana','Ragin Cajuns')",
    "INSERT INTO teams (team,nickname) values ('Missouri St.','Bears')",
    "INSERT INTO teams (team,nickname) values ('Kennesaw St.','Owls')",
    "INSERT INTO teams (team,nickname) values ('Memphis','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Oklahoma','Sooners')",
    "INSERT INTO teams (team,nickname) values ('Texas A&M','Aggies')",
    "INSERT INTO teams (team,nickname) values ('Ole Miss','Rebels')",
    "INSERT INTO teams (team,nickname) values ('Oregon','Ducks')",
    "INSERT INTO teams (team,nickname) values ('Washington St.','Cougars')",
    "INSERT INTO teams (team,nickname) values ('Toledo','Rockets')",
    "INSERT INTO teams (team,nickname) values ('Western Ky.','Hilltoppers')",
    "INSERT INTO teams (team,nickname) values ('UNLV','Rebels')",
    "INSERT INTO teams (team,nickname) values ('Hawaii','Warriors')",
    "INSERT INTO teams (team,nickname) values ('Northwestern','Wildcats')",
    "INSERT INTO teams (team,nickname) values ('Minnesota','Golden Gophers')",
    "INSERT INTO teams (team,nickname) values ('UTSA','Roadrunners')",
    "INSERT INTO teams (team,nickname) values ('East Carolina','Pirates')",
    "INSERT INTO teams (team,nickname) values ('Clemson','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Army West Point','Black Knights')",
    "INSERT INTO teams (team,nickname) values ('BYU','Cougars')",
    "INSERT INTO teams (team,nickname) values ('Fresno St.','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('San Diego St.','Aztecs')",
    "INSERT INTO teams (team,nickname) values ('Missouri','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Houston','Cougars')",
    "INSERT INTO teams (team,nickname) values ('App State','Mountaineers')",
    "INSERT INTO teams (team,nickname) values ('Louisiana Tech','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Illinois','Fighting Illini')",
    "INSERT INTO teams (team,nickname) values ('TCU','Horned Frogs')",
    "INSERT INTO teams (team,nickname) values ('Vanderbilt','Commodores')",
    "INSERT INTO teams (team,nickname) values ('Duke','Blue Devils')",
    "INSERT INTO teams (team,nickname) values ('Texas','Longhorns')",
    "INSERT INTO teams (team,nickname) values ('Nebraska','Cornhuskers')",
    "INSERT INTO teams (team,nickname) values ('Ohio St.','Buckeyes')",
    "INSERT INTO teams (team,nickname) values ('Texas Tech','Red Raiders')",
    "INSERT INTO teams (team,nickname) values ('Indiana','Hoosiers')",
    "INSERT INTO teams (team,nickname) values ('Georgia','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Texas St.','Bobcats')",
    "INSERT INTO teams (team,nickname) values ('Cincinnati','Bearcats')",
    "INSERT INTO teams (team,nickname) values ('SMU','Mustangs')",
    "INSERT INTO teams (team,nickname) values ('Mississippi St.','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Washington','Huskies')",
    "INSERT INTO teams (team,nickname) values ('Jacksonville St.','Gamecocks')",
    "INSERT INTO teams (team,nickname) values ('South Fla.','Bulls')",
    "INSERT INTO teams (team,nickname) values ('Delaware','Fightin Blue Hens')",
    "INSERT INTO teams (team,nickname) values ('Arkansas St.','Red Wolves')",
    "INSERT INTO teams (team,nickname) values ('Western Mich.','Broncos')",
    "INSERT INTO teams (team,nickname) values ('NC State','Wolfpack')",
    "INSERT INTO teams (team,nickname) values ('Alabama','Crimson Tide')",
    "INSERT INTO teams (team,nickname) values ('Miami (FL)','Hurricanes')",
    "INSERT INTO teams (team,nickname) values ('Tulane','Green Wave')",
    "INSERT INTO teams (team,nickname) values ('James Madison','Dukes')",
    "INSERT INTO teams (team,nickname) values ('Utah St.','Aggies')",
    "INSERT INTO teams (team,nickname) values ('Louisville','Cardinals')",
    "INSERT INTO teams (team,nickname) values ('Southern Miss.','Golden Eagles')",
    "INSERT INTO teams (team,nickname) values ('Ohio','Bobcats')",
    "INSERT INTO teams (team,nickname) values ('California','Golden Bears')",
    "INSERT INTO teams (team,nickname) values ('Central Mich.','Chippewas')",
    "INSERT INTO teams (team,nickname) values ('New Mexico','Lobos')",
    "INSERT INTO teams (team,nickname) values ('FIU','Panthers')",
    "INSERT INTO teams (team,nickname) values ('Pittsburgh','Panthers')",
    "INSERT INTO teams (team,nickname) values ('Penn St.','Nittany Lions')",
    "INSERT INTO teams (team,nickname) values ('UConn','Huskies')",
    "INSERT INTO teams (team,nickname) values ('Georgia Tech','Yellow Jackets')",
    "INSERT INTO teams (team,nickname) values ('Miami (OH)','RedHawks')",
    "INSERT INTO teams (team,nickname) values ('North Texas','Mean Green')",
    "INSERT INTO teams (team,nickname) values ('Virginia','Cavaliers')",
    "INSERT INTO teams (team,nickname) values ('LSU','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Ga. Southern','Eagles')",
    "INSERT INTO teams (team,nickname) values ('Coastal Carolina','Chanticleers')",
    "INSERT INTO teams (team,nickname) values ('Tennessee','Volunteers')",
    "INSERT INTO teams (team,nickname) values ('Southern California','Trojans')",
    "INSERT INTO teams (team,nickname) values ('Iowa','Hawkeyes')",
    "INSERT INTO teams (team,nickname) values ('Arizona St.','Sun Devils')",
    "INSERT INTO teams (team,nickname) values ('Michigan','Wolverines')",
    "INSERT INTO teams (team,nickname) values ('Utah','Utes')",
    "INSERT INTO teams (team,nickname) values ('Rice','Owls')",
    "INSERT INTO teams (team,nickname) values ('Arizona','Wildcats')",
    "INSERT INTO teams (team,nickname) values ('Wake Forest','Demon Deacons')",
"INSERT INTO players (player_name, number, team_id) values ('Blake Horvath',11,1);",
"INSERT INTO players (player_name, number, team_id) values ('Maddux Madsen',12,2);",
"INSERT INTO players (player_name, number, team_id) values ('Goose Crowder',13,3);",
"INSERT INTO players (player_name, number, team_id) values ('Grant Wilson',14,4);",
"INSERT INTO players (player_name, number, team_id) values ('Walker Howard',15,5);",
"INSERT INTO players (player_name, number, team_id) values ('Jacob Clark',16,6);",
"INSERT INTO players (player_name, number, team_id) values ('Dexter Williams II',17,7);",
"INSERT INTO players (player_name, number, team_id) values ('Keilon Elder',18,8);",
"INSERT INTO players (player_name, number, team_id) values ('John Mateer',19,9);",
"INSERT INTO players (player_name, number, team_id) values ('Marcel Reed',20,10);",
"INSERT INTO players (player_name, number, team_id) values ('Trinidad Chambliss',21,11);",
"INSERT INTO players (player_name, number, team_id) values ('Dante Moore',22,12);",
"INSERT INTO players (player_name, number, team_id) values ('Max Cutforth',23,13);",
"INSERT INTO players (player_name, number, team_id) values ('Tucker Gleason',24,14);",
"INSERT INTO players (player_name, number, team_id) values ('Maverick McIvor',25,15);",
"INSERT INTO players (player_name, number, team_id) values ('Anthony Colandrea',26,16);",
"INSERT INTO players (player_name, number, team_id) values ('Brayden Schager',27,17);",
"INSERT INTO players (player_name, number, team_id) values ('Jack Lausch',28,18);",
"INSERT INTO players (player_name, number, team_id) values ('Drake Lindsey',29,19);",
"INSERT INTO players (player_name, number, team_id) values ('Owen McCown',30,20);",
"INSERT INTO players (player_name, number, team_id) values ('Mason Houser',31,21);",
"INSERT INTO players (player_name, number, team_id) values ('Cade Klubnik',32,22);",
"INSERT INTO players (player_name, number, team_id) values ('Trent Reid',33,23);",
"INSERT INTO players (player_name, number, team_id) values ('Jake Retzlaff',34,24);",
"INSERT INTO players (player_name, number, team_id) values ('E.J. Smith',35,25);",
"INSERT INTO players (player_name, number, team_id) values ('Danny ONeil',36,26);",
"INSERT INTO players (player_name, number, team_id) values ('Beau Pribula',37,27);",
"INSERT INTO players (player_name, number, team_id) values ('Conner Weigman',38,28);",
"INSERT INTO players (player_name, number, team_id) values ('Jalon Jones',39,29);",
"INSERT INTO players (player_name, number, team_id) values ('Jack Turner',40,30);",
"INSERT INTO players (player_name, number, team_id) values ('Luke Altmyer',41,31);",
"INSERT INTO players (player_name, number, team_id) values ('Josh Hoover',42,32);",
"INSERT INTO players (player_name, number, team_id) values ('Diego Pavia',43,33);",
"INSERT INTO players (player_name, number, team_id) values ('Maalik Murphy',44,34);",
"INSERT INTO players (player_name, number, team_id) values ('Arch Manning',45,35);",
"INSERT INTO players (player_name, number, team_id) values ('Dylan Raiola',46,36);",
"INSERT INTO players (player_name, number, team_id) values ('Julian Sayin',47,37);",
"INSERT INTO players (player_name, number, team_id) values ('Behren Morton',48,38);",
"INSERT INTO players (player_name, number, team_id) values ('Fernando Mendoza',49,39);",
"INSERT INTO players (player_name, number, team_id) values ('Gunner Stockton',50,40);",
"INSERT INTO players (player_name, number, team_id) values ('Jordan McCloud',51,41);",
"INSERT INTO players (player_name, number, team_id) values ('Myles Sorsby',52,42);",
"INSERT INTO players (player_name, number, team_id) values ('Kevin Jennings',53,43);",
"INSERT INTO players (player_name, number, team_id) values ('Blake Shapen',54,44);",
"INSERT INTO players (player_name, number, team_id) values ('Demond Williams Jr.',55,45);",
"INSERT INTO players (player_name, number, team_id) values ('Caden Creel',56,46);",
"INSERT INTO players (player_name, number, team_id) values ('Byrum Brown',57,47);",
"INSERT INTO players (player_name, number, team_id) values ('Nolan Henderson',58,48);",
"INSERT INTO players (player_name, number, team_id) values ('Jaylen Raynor',59,49);",
"INSERT INTO players (player_name, number, team_id) values ('Hayden Wolff',60,50);",
"INSERT INTO players (player_name, number, team_id) values ('Tamarcus Cooley',61,51);",
"INSERT INTO players (player_name, number, team_id) values ('Ty Simpson',62,52);",
"INSERT INTO players (player_name, number, team_id) values ('Carson Beck',63,53);",
"INSERT INTO players (player_name, number, team_id) values ('Jake Retzlaff',64,54);",
"INSERT INTO players (player_name, number, team_id) values ('Alonza Barnett III',65,55);",
"INSERT INTO players (player_name, number, team_id) values ('Bryson Barnes',66,56);",
"INSERT INTO players (player_name, number, team_id) values ('Tyler Shough',67,57);",
"INSERT INTO players (player_name, number, team_id) values ('Zach Wilcke',68,58);",
"INSERT INTO players (player_name, number, team_id) values ('Parker Navarro',69,59);",
"INSERT INTO players (player_name, number, team_id) values ('Jaron-Keawe Sagapolutele',70,60);",
"INSERT INTO players (player_name, number, team_id) values ('Bert Emanuel Jr.',71,61);",
"INSERT INTO players (player_name, number, team_id) values ('Devin Dampier',72,62);",
"INSERT INTO players (player_name, number, team_id) values ('Kenji Bahar',73,63);",
"INSERT INTO players (player_name, number, team_id) values ('Eli Holstein',74,64);",
"INSERT INTO players (player_name, number, team_id) values ('Drew Allar',75,65);",
"INSERT INTO players (player_name, number, team_id) values ('Zion Turner',76,66);",
"INSERT INTO players (player_name, number, team_id) values ('Haynes King',77,67);",
"INSERT INTO players (player_name, number, team_id) values ('Brett Gabbert',78,68);",
"INSERT INTO players (player_name, number, team_id) values ('Chandler Morris',79,69);",
"INSERT INTO players (player_name, number, team_id) values ('Tony Muskett',80,70);",
"INSERT INTO players (player_name, number, team_id) values ('Garrett Nussmeier',81,71);",
"INSERT INTO players (player_name, number, team_id) values ('JC French',82,72);",
"INSERT INTO players (player_name, number, team_id) values ('Jaylen Henderson',83,73);",
"INSERT INTO players (player_name, number, team_id) values ('Joey Aguilar',84,74);",
"INSERT INTO players (player_name, number, team_id) values ('Jayden Maiava',85,75);",
"INSERT INTO players (player_name, number, team_id) values ('Riley Moss',86,76);",
"INSERT INTO players (player_name, number, team_id) values ('Sam Leavitt',87,77);",
"INSERT INTO players (player_name, number, team_id) values ('Bryce Underwood',88,78);",
"INSERT INTO players (player_name, number, team_id) values ('Cameron Rising',89,79);",
"INSERT INTO players (player_name, number, team_id) values ('JT Grisham',90,80);",
"INSERT INTO players (player_name, number, team_id) values ('Noah Fifita',91,81);",
"INSERT INTO players (player_name, number, team_id) values ('Mitch Griffis',92,82);",

"INSERT INTO coaches (coach_name, team_id) values ('Brian Newberry',1);",
"INSERT INTO coaches (coach_name, team_id) values ('Spencer Danielson',2);",
"INSERT INTO coaches (coach_name, team_id) values ('Trent Miles',3);",
"INSERT INTO coaches (coach_name, team_id) values ('Ricky Rahne',4);",
"INSERT INTO coaches (coach_name, team_id) values ('Michael Desormeaux',5);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Beard',6);",
"INSERT INTO coaches (coach_name, team_id) values ('Brian Bohannon',7);",
"INSERT INTO coaches (coach_name, team_id) values ('Kevin Wright',8);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Venables',9);",
"INSERT INTO coaches (coach_name, team_id) values ('Mike Elko',10);",
"INSERT INTO coaches (coach_name, team_id) values ('Pete Golding',11);",
"INSERT INTO coaches (coach_name, team_id) values ('Dan Lanning',12);",
"INSERT INTO coaches (coach_name, team_id) values ('Jimmy Rogers',13);",
"INSERT INTO coaches (coach_name, team_id) values ('Ben Albert',14);",
"INSERT INTO coaches (coach_name, team_id) values ('Cord Sandberg',15);",
"INSERT INTO coaches (coach_name, team_id) values ('Dan Mullen',16);",
"INSERT INTO coaches (coach_name, team_id) values ('Billy Dormandy',17);",
"INSERT INTO coaches (coach_name, team_id) values ('David Braun',18);",
"INSERT INTO coaches (coach_name, team_id) values ('P.J. Fleck',19);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Traylor',20);",
"INSERT INTO coaches (coach_name, team_id) values ('Curt Cignetti',21);",
"INSERT INTO coaches (coach_name, team_id) values ('Dabo Swinney',22);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Monken',23);",
"INSERT INTO coaches (coach_name, team_id) values ('Kalani Sitake',24);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Entz',25);",
"INSERT INTO coaches (coach_name, team_id) values ('Sean Lewis',26);",
"INSERT INTO coaches (coach_name, team_id) values ('Eli Drinkwitz',27);",
"INSERT INTO coaches (coach_name, team_id) values ('Willie Fritz',28);",
"INSERT INTO coaches (coach_name, team_id) values ('James Fortune',29);",
"INSERT INTO coaches (coach_name, team_id) values ('Sonny Cumbie',30);",
"INSERT INTO coaches (coach_name, team_id) values ('Bret Bielema',31);",
"INSERT INTO coaches (coach_name, team_id) values ('Sonny Dykes',32);",
"INSERT INTO coaches (coach_name, team_id) values ('Clark Lea',33);",
"INSERT INTO coaches (coach_name, team_id) values ('Manny Diaz',34);",
"INSERT INTO coaches (coach_name, team_id) values ('Steve Sarkisian',35);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Rhule',36);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Day',37);",
"INSERT INTO coaches (coach_name, team_id) values ('Joey McGuire',38);",
"INSERT INTO coaches (coach_name, team_id) values ('Curt Cignetti',39);",
"INSERT INTO coaches (coach_name, team_id) values ('Kirby Smart',40);",
"INSERT INTO coaches (coach_name, team_id) values ('G.J. Kinne',41);",
"INSERT INTO coaches (coach_name, team_id) values ('Scott Satterfield',42);",
"INSERT INTO coaches (coach_name, team_id) values ('Rhett Lashlee',43);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Lebby',44);",
"INSERT INTO coaches (coach_name, team_id) values ('Jedd Fisch',45);",
"INSERT INTO coaches (coach_name, team_id) values ('Rich Rodriguez Jr.',46);",
"INSERT INTO coaches (coach_name, team_id) values ('Hank Fourcade',47);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Carty',48);",
"INSERT INTO coaches (coach_name, team_id) values ('Butch Jones',49);",
"INSERT INTO coaches (coach_name, team_id) values ('Lance Taylor',50);",
"INSERT INTO coaches (coach_name, team_id) values ('Dave Doeren',51);",
"INSERT INTO coaches (coach_name, team_id) values ('Kalen DeBoer',52);",
"INSERT INTO coaches (coach_name, team_id) values ('Mario Cristobal',53);",
"INSERT INTO coaches (coach_name, team_id) values ('Jon Sumrall',54);",
"INSERT INTO coaches (coach_name, team_id) values ('Bob Chesney',55);",
"INSERT INTO coaches (coach_name, team_id) values ('Bronco Mendenhall',56);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Brohm',57);",
"INSERT INTO coaches (coach_name, team_id) values ('Scott Satterfield',58);",
"INSERT INTO coaches (coach_name, team_id) values ('Tim Albin',59);",
"INSERT INTO coaches (coach_name, team_id) values ('Justin Wilcox',60);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Drinkall',61);",
"INSERT INTO coaches (coach_name, team_id) values ('Bronco Mendenhall',62);",
"INSERT INTO coaches (coach_name, team_id) values ('Skyler Fulton',63);",
"INSERT INTO coaches (coach_name, team_id) values ('Pat Narduzzi',64);",
"INSERT INTO coaches (coach_name, team_id) values ('James Franklin',65);",
"INSERT INTO coaches (coach_name, team_id) values ('Jim Mora',66);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Key',67);",
"INSERT INTO coaches (coach_name, team_id) values ('Chuck Martin',68);",
"INSERT INTO coaches (coach_name, team_id) values ('Eric Morris',69);",
"INSERT INTO coaches (coach_name, team_id) values ('Tony Elliott',70);",
"INSERT INTO coaches (coach_name, team_id) values ('Brian Kelly',71);",
"INSERT INTO coaches (coach_name, team_id) values ('Clay Helton',72);",
"INSERT INTO coaches (coach_name, team_id) values ('Chadwick Saddler',73);",
"INSERT INTO coaches (coach_name, team_id) values ('Josh Heupel',74);",
"INSERT INTO coaches (coach_name, team_id) values ('Lincoln Riley',75);",
"INSERT INTO coaches (coach_name, team_id) values ('Kirk Ferentz',76);",
"INSERT INTO coaches (coach_name, team_id) values ('Kenny Dillingham',77);",
"INSERT INTO coaches (coach_name, team_id) values ('Sherrone Moore',78);",
"INSERT INTO coaches (coach_name, team_id) values ('Morgan Scalley',79);",
"INSERT INTO coaches (coach_name, team_id) values ('AJ Miller',80);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Brennan',81);",
"INSERT INTO coaches (coach_name, team_id) values ('Jake Dickert',82);",
]



for statement in insert_statements:
    cursor.execute(statement)



conn.commit()
conn.close()

print("Database created!")
```



In [ ]:

import sqlite3
conn = sqlite3.connect('teams.db')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE teams (
        team_id INTEGER PRIMARY KEY AUTOINCREMENT,
        team TEXT,
        nickname TEXT)''')

cursor.execute('''CREATE TABLE players (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_name TEXT NOT NULL,
        number INTEGER,
        team_id INTEGER,
        FOREIGN KEY (team_id) REFERENCES teams(team_id))''')

cursor.execute('''CREATE TABLE coaches (
        coach_id INTEGER PRIMARY KEY AUTOINCREMENT,
        coach_name TEXT NOT NULL,
        team_id INTEGER,
        FOREIGN KEY (team_id) REFERENCES teams(team_id))''')

insert_statements = [
    "INSERT INTO teams (team,nickname) values ('Troy','Trojans')",
    "INSERT INTO teams (team,nickname) values ('Old Dominion','Monarchs')",
    "INSERT INTO teams (team,nickname) values ('Louisiana','Ragin Cajuns')",
    "INSERT INTO teams (team,nickname) values ('Missouri St.','Bears')",
    "INSERT INTO teams (team,nickname) values ('Kennesaw St.','Owls')",
    "INSERT INTO teams (team,nickname) values ('Memphis','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Oklahoma','Sooners')",
    "INSERT INTO teams (team,nickname) values ('Texas A&M','Aggies')",
    "INSERT INTO teams (team,nickname) values ('Ole Miss','Rebels')",
    "INSERT INTO teams (team,nickname) values ('Oregon','Ducks')",
    "INSERT INTO teams (team,nickname) values ('Washington St.','Cougars')",
    "INSERT INTO teams (team,nickname) values ('Toledo','Rockets')",
    "INSERT INTO teams (team,nickname) values ('Western Ky.','Hilltoppers')",
    "INSERT INTO teams (team,nickname) values ('UNLV','Rebels')",
    "INSERT INTO teams (team,nickname) values ('Hawaii','Warriors')",
    "INSERT INTO teams (team,nickname) values ('Northwestern','Wildcats')",
    "INSERT INTO teams (team,nickname) values ('Minnesota','Golden Gophers')",
    "INSERT INTO teams (team,nickname) values ('UTSA','Roadrunners')",
    "INSERT INTO teams (team,nickname) values ('East Carolina','Pirates')",
    "INSERT INTO teams (team,nickname) values ('Clemson','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Army West Point','Black Knights')",
    "INSERT INTO teams (team,nickname) values ('BYU','Cougars')",
    "INSERT INTO teams (team,nickname) values ('Fresno St.','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('San Diego St.','Aztecs')",
    "INSERT INTO teams (team,nickname) values ('Missouri','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Houston','Cougars')",
    "INSERT INTO teams (team,nickname) values ('App State','Mountaineers')",
    "INSERT INTO teams (team,nickname) values ('Louisiana Tech','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Illinois','Fighting Illini')",
    "INSERT INTO teams (team,nickname) values ('TCU','Horned Frogs')",
    "INSERT INTO teams (team,nickname) values ('Vanderbilt','Commodores')",
    "INSERT INTO teams (team,nickname) values ('Duke','Blue Devils')",
    "INSERT INTO teams (team,nickname) values ('Texas','Longhorns')",
    "INSERT INTO teams (team,nickname) values ('Nebraska','Cornhuskers')",
    "INSERT INTO teams (team,nickname) values ('Ohio St.','Buckeyes')",
    "INSERT INTO teams (team,nickname) values ('Texas Tech','Red Raiders')",
    "INSERT INTO teams (team,nickname) values ('Indiana','Hoosiers')",
    "INSERT INTO teams (team,nickname) values ('Georgia','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Texas St.','Bobcats')",
    "INSERT INTO teams (team,nickname) values ('Cincinnati','Bearcats')",
    "INSERT INTO teams (team,nickname) values ('SMU','Mustangs')",
    "INSERT INTO teams (team,nickname) values ('Mississippi St.','Bulldogs')",
    "INSERT INTO teams (team,nickname) values ('Washington','Huskies')",
    "INSERT INTO teams (team,nickname) values ('Jacksonville St.','Gamecocks')",
    "INSERT INTO teams (team,nickname) values ('South Fla.','Bulls')",
    "INSERT INTO teams (team,nickname) values ('Delaware','Fightin Blue Hens')",
    "INSERT INTO teams (team,nickname) values ('Arkansas St.','Red Wolves')",
    "INSERT INTO teams (team,nickname) values ('Western Mich.','Broncos')",
    "INSERT INTO teams (team,nickname) values ('NC State','Wolfpack')",
    "INSERT INTO teams (team,nickname) values ('Alabama','Crimson Tide')",
    "INSERT INTO teams (team,nickname) values ('Miami (FL)','Hurricanes')",
    "INSERT INTO teams (team,nickname) values ('Tulane','Green Wave')",
    "INSERT INTO teams (team,nickname) values ('James Madison','Dukes')",
    "INSERT INTO teams (team,nickname) values ('Utah St.','Aggies')",
    "INSERT INTO teams (team,nickname) values ('Louisville','Cardinals')",
    "INSERT INTO teams (team,nickname) values ('Southern Miss.','Golden Eagles')",
    "INSERT INTO teams (team,nickname) values ('Ohio','Bobcats')",
    "INSERT INTO teams (team,nickname) values ('California','Golden Bears')",
    "INSERT INTO teams (team,nickname) values ('Central Mich.','Chippewas')",
    "INSERT INTO teams (team,nickname) values ('New Mexico','Lobos')",
    "INSERT INTO teams (team,nickname) values ('FIU','Panthers')",
    "INSERT INTO teams (team,nickname) values ('Pittsburgh','Panthers')",
    "INSERT INTO teams (team,nickname) values ('Penn St.','Nittany Lions')",
    "INSERT INTO teams (team,nickname) values ('UConn','Huskies')",
    "INSERT INTO teams (team,nickname) values ('Georgia Tech','Yellow Jackets')",
    "INSERT INTO teams (team,nickname) values ('Miami (OH)','RedHawks')",
    "INSERT INTO teams (team,nickname) values ('North Texas','Mean Green')",
    "INSERT INTO teams (team,nickname) values ('Virginia','Cavaliers')",
    "INSERT INTO teams (team,nickname) values ('LSU','Tigers')",
    "INSERT INTO teams (team,nickname) values ('Ga. Southern','Eagles')",
    "INSERT INTO teams (team,nickname) values ('Coastal Carolina','Chanticleers')",
    "INSERT INTO teams (team,nickname) values ('Tennessee','Volunteers')",
    "INSERT INTO teams (team,nickname) values ('Southern California','Trojans')",
    "INSERT INTO teams (team,nickname) values ('Iowa','Hawkeyes')",
    "INSERT INTO teams (team,nickname) values ('Arizona St.','Sun Devils')",
    "INSERT INTO teams (team,nickname) values ('Michigan','Wolverines')",
    "INSERT INTO teams (team,nickname) values ('Utah','Utes')",
    "INSERT INTO teams (team,nickname) values ('Rice','Owls')",
    "INSERT INTO teams (team,nickname) values ('Arizona','Wildcats')",
    "INSERT INTO teams (team,nickname) values ('Wake Forest','Demon Deacons')",
    "INSERT INTO teams (team,nickname) values ('Navy','Mids')",

"INSERT INTO players (player_name, number, team_id) values ('Blake Horvath',11,1);",
"INSERT INTO players (player_name, number, team_id) values ('Maddux Madsen',12,2);",
"INSERT INTO players (player_name, number, team_id) values ('Goose Crowder',13,3);",
"INSERT INTO players (player_name, number, team_id) values ('Grant Wilson',14,4);",
"INSERT INTO players (player_name, number, team_id) values ('Walker Howard',15,5);",
"INSERT INTO players (player_name, number, team_id) values ('Jacob Clark',16,6);",
"INSERT INTO players (player_name, number, team_id) values ('Dexter Williams II',17,7);",
"INSERT INTO players (player_name, number, team_id) values ('Keilon Elder',18,8);",
"INSERT INTO players (player_name, number, team_id) values ('John Mateer',19,9);",
"INSERT INTO players (player_name, number, team_id) values ('Marcel Reed',20,10);",
"INSERT INTO players (player_name, number, team_id) values ('Trinidad Chambliss',21,11);",
"INSERT INTO players (player_name, number, team_id) values ('Dante Moore',22,12);",
"INSERT INTO players (player_name, number, team_id) values ('Max Cutforth',23,13);",
"INSERT INTO players (player_name, number, team_id) values ('Tucker Gleason',24,14);",
"INSERT INTO players (player_name, number, team_id) values ('Maverick McIvor',25,15);",
"INSERT INTO players (player_name, number, team_id) values ('Anthony Colandrea',26,16);",
"INSERT INTO players (player_name, number, team_id) values ('Brayden Schager',27,17);",
"INSERT INTO players (player_name, number, team_id) values ('Jack Lausch',28,18);",
"INSERT INTO players (player_name, number, team_id) values ('Drake Lindsey',29,19);",
"INSERT INTO players (player_name, number, team_id) values ('Owen McCown',30,20);",
"INSERT INTO players (player_name, number, team_id) values ('Mason Houser',31,21);",
"INSERT INTO players (player_name, number, team_id) values ('Cade Klubnik',32,22);",
"INSERT INTO players (player_name, number, team_id) values ('Trent Reid',33,23);",
"INSERT INTO players (player_name, number, team_id) values ('Jake Retzlaff',34,24);",
"INSERT INTO players (player_name, number, team_id) values ('E.J. Smith',35,25);",
"INSERT INTO players (player_name, number, team_id) values ('Danny ONeil',36,26);",
"INSERT INTO players (player_name, number, team_id) values ('Beau Pribula',37,27);",
"INSERT INTO players (player_name, number, team_id) values ('Conner Weigman',38,28);",
"INSERT INTO players (player_name, number, team_id) values ('Jalon Jones',39,29);",
"INSERT INTO players (player_name, number, team_id) values ('Jack Turner',40,30);",
"INSERT INTO players (player_name, number, team_id) values ('Luke Altmyer',41,31);",
"INSERT INTO players (player_name, number, team_id) values ('Josh Hoover',42,32);",
"INSERT INTO players (player_name, number, team_id) values ('Diego Pavia',43,33);",
"INSERT INTO players (player_name, number, team_id) values ('Maalik Murphy',44,34);",
"INSERT INTO players (player_name, number, team_id) values ('Arch Manning',45,35);",
"INSERT INTO players (player_name, number, team_id) values ('Dylan Raiola',46,36);",
"INSERT INTO players (player_name, number, team_id) values ('Julian Sayin',47,37);",
"INSERT INTO players (player_name, number, team_id) values ('Behren Morton',48,38);",
"INSERT INTO players (player_name, number, team_id) values ('Fernando Mendoza',49,39);",
"INSERT INTO players (player_name, number, team_id) values ('Gunner Stockton',50,40);",
"INSERT INTO players (player_name, number, team_id) values ('Jordan McCloud',51,41);",
"INSERT INTO players (player_name, number, team_id) values ('Myles Sorsby',52,42);",
"INSERT INTO players (player_name, number, team_id) values ('Kevin Jennings',53,43);",
"INSERT INTO players (player_name, number, team_id) values ('Blake Shapen',54,44);",
"INSERT INTO players (player_name, number, team_id) values ('Demond Williams Jr.',55,45);",
"INSERT INTO players (player_name, number, team_id) values ('Caden Creel',56,46);",
"INSERT INTO players (player_name, number, team_id) values ('Byrum Brown',57,47);",
"INSERT INTO players (player_name, number, team_id) values ('Nolan Henderson',58,48);",
"INSERT INTO players (player_name, number, team_id) values ('Jaylen Raynor',59,49);",
"INSERT INTO players (player_name, number, team_id) values ('Hayden Wolff',60,50);",
"INSERT INTO players (player_name, number, team_id) values ('Tamarcus Cooley',61,51);",
"INSERT INTO players (player_name, number, team_id) values ('Ty Simpson',62,52);",
"INSERT INTO players (player_name, number, team_id) values ('Carson Beck',63,53);",
"INSERT INTO players (player_name, number, team_id) values ('Jake Retzlaff',64,54);",
"INSERT INTO players (player_name, number, team_id) values ('Alonza Barnett III',65,55);",
"INSERT INTO players (player_name, number, team_id) values ('Bryson Barnes',66,56);",
"INSERT INTO players (player_name, number, team_id) values ('Tyler Shough',67,57);",
"INSERT INTO players (player_name, number, team_id) values ('Zach Wilcke',68,58);",
"INSERT INTO players (player_name, number, team_id) values ('Parker Navarro',69,59);",
"INSERT INTO players (player_name, number, team_id) values ('Jaron-Keawe Sagapolutele',70,60);",
"INSERT INTO players (player_name, number, team_id) values ('Bert Emanuel Jr.',71,61);",
"INSERT INTO players (player_name, number, team_id) values ('Devin Dampier',72,62);",
"INSERT INTO players (player_name, number, team_id) values ('Kenji Bahar',73,63);",
"INSERT INTO players (player_name, number, team_id) values ('Eli Holstein',74,64);",
"INSERT INTO players (player_name, number, team_id) values ('Drew Allar',75,65);",
"INSERT INTO players (player_name, number, team_id) values ('Zion Turner',76,66);",
"INSERT INTO players (player_name, number, team_id) values ('Haynes King',77,67);",
"INSERT INTO players (player_name, number, team_id) values ('Brett Gabbert',78,68);",
"INSERT INTO players (player_name, number, team_id) values ('Chandler Morris',79,69);",
"INSERT INTO players (player_name, number, team_id) values ('Tony Muskett',80,70);",
"INSERT INTO players (player_name, number, team_id) values ('Garrett Nussmeier',81,71);",
"INSERT INTO players (player_name, number, team_id) values ('JC French',82,72);",
"INSERT INTO players (player_name, number, team_id) values ('Jaylen Henderson',83,73);",
"INSERT INTO players (player_name, number, team_id) values ('Joey Aguilar',84,74);",
"INSERT INTO players (player_name, number, team_id) values ('Jayden Maiava',85,75);",
"INSERT INTO players (player_name, number, team_id) values ('Riley Moss',86,76);",
"INSERT INTO players (player_name, number, team_id) values ('Sam Leavitt',87,77);",
"INSERT INTO players (player_name, number, team_id) values ('Bryce Underwood',88,78);",
"INSERT INTO players (player_name, number, team_id) values ('Cameron Rising',89,79);",
"INSERT INTO players (player_name, number, team_id) values ('JT Grisham',90,80);",
"INSERT INTO players (player_name, number, team_id) values ('Noah Fifita',91,81);",
"INSERT INTO players (player_name, number, team_id) values ('Mitch Griffis',92,82);",
"INSERT INTO players (player_name, number, team_id) values ('Blake Horvath',99,83);",

"INSERT INTO coaches (coach_name, team_id) values ('Brian Newberry',1);",
"INSERT INTO coaches (coach_name, team_id) values ('Spencer Danielson',2);",
"INSERT INTO coaches (coach_name, team_id) values ('Trent Miles',3);",
"INSERT INTO coaches (coach_name, team_id) values ('Ricky Rahne',4);",
"INSERT INTO coaches (coach_name, team_id) values ('Michael Desormeaux',5);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Beard',6);",
"INSERT INTO coaches (coach_name, team_id) values ('Brian Bohannon',7);",
"INSERT INTO coaches (coach_name, team_id) values ('Kevin Wright',8);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Venables',9);",
"INSERT INTO coaches (coach_name, team_id) values ('Mike Elko',10);",
"INSERT INTO coaches (coach_name, team_id) values ('Pete Golding',11);",
"INSERT INTO coaches (coach_name, team_id) values ('Dan Lanning',12);",
"INSERT INTO coaches (coach_name, team_id) values ('Jimmy Rogers',13);",
"INSERT INTO coaches (coach_name, team_id) values ('Ben Albert',14);",
"INSERT INTO coaches (coach_name, team_id) values ('Cord Sandberg',15);",
"INSERT INTO coaches (coach_name, team_id) values ('Dan Mullen',16);",
"INSERT INTO coaches (coach_name, team_id) values ('Billy Dormandy',17);",
"INSERT INTO coaches (coach_name, team_id) values ('David Braun',18);",
"INSERT INTO coaches (coach_name, team_id) values ('P.J. Fleck',19);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Traylor',20);",
"INSERT INTO coaches (coach_name, team_id) values ('Curt Cignetti',21);",
"INSERT INTO coaches (coach_name, team_id) values ('Dabo Swinney',22);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Monken',23);",
"INSERT INTO coaches (coach_name, team_id) values ('Kalani Sitake',24);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Entz',25);",
"INSERT INTO coaches (coach_name, team_id) values ('Sean Lewis',26);",
"INSERT INTO coaches (coach_name, team_id) values ('Eli Drinkwitz',27);",
"INSERT INTO coaches (coach_name, team_id) values ('Willie Fritz',28);",
"INSERT INTO coaches (coach_name, team_id) values ('James Fortune',29);",
"INSERT INTO coaches (coach_name, team_id) values ('Sonny Cumbie',30);",
"INSERT INTO coaches (coach_name, team_id) values ('Bret Bielema',31);",
"INSERT INTO coaches (coach_name, team_id) values ('Sonny Dykes',32);",
"INSERT INTO coaches (coach_name, team_id) values ('Clark Lea',33);",
"INSERT INTO coaches (coach_name, team_id) values ('Manny Diaz',34);",
"INSERT INTO coaches (coach_name, team_id) values ('Steve Sarkisian',35);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Rhule',36);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Day',37);",
"INSERT INTO coaches (coach_name, team_id) values ('Joey McGuire',38);",
"INSERT INTO coaches (coach_name, team_id) values ('Curt Cignetti',39);",
"INSERT INTO coaches (coach_name, team_id) values ('Kirby Smart',40);",
"INSERT INTO coaches (coach_name, team_id) values ('G.J. Kinne',41);",
"INSERT INTO coaches (coach_name, team_id) values ('Scott Satterfield',42);",
"INSERT INTO coaches (coach_name, team_id) values ('Rhett Lashlee',43);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Lebby',44);",
"INSERT INTO coaches (coach_name, team_id) values ('Jedd Fisch',45);",
"INSERT INTO coaches (coach_name, team_id) values ('Rich Rodriguez Jr.',46);",
"INSERT INTO coaches (coach_name, team_id) values ('Hank Fourcade',47);",
"INSERT INTO coaches (coach_name, team_id) values ('Ryan Carty',48);",
"INSERT INTO coaches (coach_name, team_id) values ('Butch Jones',49);",
"INSERT INTO coaches (coach_name, team_id) values ('Lance Taylor',50);",
"INSERT INTO coaches (coach_name, team_id) values ('Dave Doeren',51);",
"INSERT INTO coaches (coach_name, team_id) values ('Kalen DeBoer',52);",
"INSERT INTO coaches (coach_name, team_id) values ('Mario Cristobal',53);",
"INSERT INTO coaches (coach_name, team_id) values ('Jon Sumrall',54);",
"INSERT INTO coaches (coach_name, team_id) values ('Bob Chesney',55);",
"INSERT INTO coaches (coach_name, team_id) values ('Bronco Mendenhall',56);",
"INSERT INTO coaches (coach_name, team_id) values ('Jeff Brohm',57);",
"INSERT INTO coaches (coach_name, team_id) values ('Scott Satterfield',58);",
"INSERT INTO coaches (coach_name, team_id) values ('Tim Albin',59);",
"INSERT INTO coaches (coach_name, team_id) values ('Justin Wilcox',60);",
"INSERT INTO coaches (coach_name, team_id) values ('Matt Drinkall',61);",
"INSERT INTO coaches (coach_name, team_id) values ('Bronco Mendenhall',62);",
"INSERT INTO coaches (coach_name, team_id) values ('Skyler Fulton',63);",
"INSERT INTO coaches (coach_name, team_id) values ('Pat Narduzzi',64);",
"INSERT INTO coaches (coach_name, team_id) values ('James Franklin',65);",
"INSERT INTO coaches (coach_name, team_id) values ('Jim Mora',66);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Key',67);",
"INSERT INTO coaches (coach_name, team_id) values ('Chuck Martin',68);",
"INSERT INTO coaches (coach_name, team_id) values ('Eric Morris',69);",
"INSERT INTO coaches (coach_name, team_id) values ('Tony Elliott',70);",
"INSERT INTO coaches (coach_name, team_id) values ('Brian Kelly',71);",
"INSERT INTO coaches (coach_name, team_id) values ('Clay Helton',72);",
"INSERT INTO coaches (coach_name, team_id) values ('Chadwick Saddler',73);",
"INSERT INTO coaches (coach_name, team_id) values ('Josh Heupel',74);",
"INSERT INTO coaches (coach_name, team_id) values ('Lincoln Riley',75);",
"INSERT INTO coaches (coach_name, team_id) values ('Kirk Ferentz',76);",
"INSERT INTO coaches (coach_name, team_id) values ('Kenny Dillingham',77);",
"INSERT INTO coaches (coach_name, team_id) values ('Sherrone Moore',78);",
"INSERT INTO coaches (coach_name, team_id) values ('Morgan Scalley',79);",
"INSERT INTO coaches (coach_name, team_id) values ('AJ Miller',80);",
"INSERT INTO coaches (coach_name, team_id) values ('Brent Brennan',81);",
"INSERT INTO coaches (coach_name, team_id) values ('Jake Dickert',82);",
"INSERT INTO coaches (coach_name, team_id) values ('Brian Newberry',83);"
]



for statement in insert_statements:
    cursor.execute(statement)



conn.commit()
conn.close()

print("Database created!")



Database created!
